In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import altair as alt
import matplotlib.pyplot as plt

In [2]:
#
df = pd.read_csv('train.csv')

# drop id
df = df.drop(columns=['id'])

# rename columns
df.columns = [
    'age', 'sex', 'chest_pain_type', 'bp', 'cholesterol',
    'fbs_over_120', 'ekg_results', 'max_hr', 'exercise_angina',
    'st_depression', 'slope_st', 'vessels_fluoro', 'thallium',
    'target'
]

# decode categorical columns
df['sex'] = df['sex'].map({1: 'male', 0: 'female'})
df['chest_pain_type'] = df['chest_pain_type'].map({1: 'typical', 2: 'atypical', 3: 'non-anginal',4: 'asymptomatic'})
df['exercise_angina'] = df['exercise_angina'].map({1: 'yes', 0: 'no'})
df['thallium'] = df['thallium'].map({3: 'normal', 6: 'fixed_defect', 7: 'reversible_defect'})
df['ekg_results'] = df['ekg_results'].map({0: 'normal', 1: 'st_t_abnormality',2: 'lv_hypertrophy'})
df['slope_st'] = df['slope_st'].map({1: 'upsloping', 2: 'flat', 3: 'downsloping'})
df['fbs_over_120'] = df['fbs_over_120'].map({1: 'yes', 0: 'no'})

# remove duplicates
df = df.drop_duplicates()

# Handle missing values (if any)
df = df.dropna()

# remove outlier rows using IQR on continuous columns
for col in ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

# reset index after row drops
df = df.reset_index(drop=True)

# set proper dtypes for categoricals
cat_cols = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
            'exercise_angina', 'slope_st', 'thallium']
df[cat_cols] = df[cat_cols].astype('category')

print("Final shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nPreview:\n", df.head())
print("\nStats:\n", df.describe())

Final shape: (594444, 14)

Columns: ['age', 'sex', 'chest_pain_type', 'bp', 'cholesterol', 'fbs_over_120', 'ekg_results', 'max_hr', 'exercise_angina', 'st_depression', 'slope_st', 'vessels_fluoro', 'thallium', 'target']

Missing values:
 age                0
sex                0
chest_pain_type    0
bp                 0
cholesterol        0
fbs_over_120       0
ekg_results        0
max_hr             0
exercise_angina    0
st_depression      0
slope_st           0
vessels_fluoro     0
thallium           0
target             0
dtype: int64

Preview:
    age     sex chest_pain_type   bp  cholesterol fbs_over_120     ekg_results  \
0   52    male         typical  125          325           no  lv_hypertrophy   
1   56  female        atypical  160          188           no  lv_hypertrophy   
2   44  female     non-anginal  134          229           no  lv_hypertrophy   
3   38    male    asymptomatic  138          283           no          normal   
4   59    male    asymptomatic  130    

In [3]:
from sklearn.model_selection import train_test_split

# Stratified sampling (8000)

df_sample, _ = train_test_split(
    df,
    train_size=8000,
    stratify=df['target'],
    random_state=42
)

df_sample = df_sample.reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print("\nClass distribution:\n", df_sample['target'].value_counts(normalize=True))

Sample shape: (8000, 14)

Class distribution:
 target
Absence     0.56875
Presence    0.43125
Name: proportion, dtype: float64


In [4]:
# Set up

# Altair row limit
alt.data_transformers.disable_max_rows()

# create a readable disease label column
df_sample['disease'] = df_sample['target'].map({
    'Absence': 'No Disease',
    'Presence': 'Disease'
})

In [6]:
# Visualization 1:
# Histogram: Age distribution by disease status

hist = alt.Chart(df_sample).mark_bar(opacity=0.6).encode(
    x=alt.X('age:Q', bin=alt.Bin(maxbins=25), title='Age'),
    y=alt.Y('count():Q', title='Count'),
    color=alt.Color(
        'disease:N',
        title='Disease Status',
        scale=alt.Scale(domain=['No Disease', 'Disease'])
    ),
    tooltip=[
        alt.Tooltip('disease:N', title='Disease Status'),
        alt.Tooltip('count():Q', title='Count')
    ]
).properties(
    title='Age Distribution by Heart Disease Status',
    width=500,
    height=350
).interactive()
hist

alt.Chart(...)

In [7]:
hist.save("hist.html")

In [8]:
# Visualization 2:
# Bar chart: Heart disease rate by chest pain type (with sex filter)

# numeric version for rate calculation
df_sample['target_num'] = df_sample['target'].map({
    'Absence': 0,
    'Presence': 1
})

# dropdown filter for sex
sex_dropdown = alt.binding_select(
    options=['All', 'male', 'female'],
    name='Sex: '
)

sex_select = alt.param(
    name='selected_sex',
    value='All',
    bind=sex_dropdown
)

bar = alt.Chart(df_sample).add_params(
    sex_select
).transform_filter(
    "(selected_sex == 'All') || datum.sex == selected_sex"
).transform_aggregate(
    total='count()',
    disease_cases='sum(target_num)',
    groupby=['chest_pain_type']
).transform_calculate(
    disease_rate='datum.disease_cases / datum.total'
).mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3).encode(
    x=alt.X(
        'chest_pain_type:N',
        sort='-y',
        title='Chest Pain Type',
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        'disease_rate:Q',
        title='Heart Disease Rate',
        axis=alt.Axis(format='%')
    ),
    color=alt.Color(
        'chest_pain_type:N',
        title='Chest Pain Type',
        legend=None
    ),
    tooltip=[
        alt.Tooltip('chest_pain_type:N', title='Chest Pain Type'),
        alt.Tooltip('disease_rate:Q', title='Disease Rate', format='.2%'),
        alt.Tooltip('total:Q', title='Count')
    ]
).properties(
    title='Heart Disease Rate by Chest Pain Type',
    width=500,
    height=350
)

bar
bar.save("bar.html")

In [ ]:
# Visualization 3:
# Scatter Plot: max_hr vs st_depression, color by disease

brush = alt.selection_interval()

points = alt.Chart(df_sample).mark_circle(size=60, opacity=0.35).encode(
    x=alt.X('max_hr:Q', title='Maximum Heart Rate'),
    y=alt.Y('st_depression:Q', title='ST Depression'),
    color=alt.condition(
        brush,
        alt.Color(
            'disease:N',
            title='Disease Status',
            scale=alt.Scale(domain=['No Disease', 'Disease'])
        ),
        alt.value('lightgray')
    ),
    tooltip=[
        alt.Tooltip('age:Q', title='Age'),
        alt.Tooltip('sex:N', title='Sex'),
        alt.Tooltip('max_hr:Q', title='Maximum Heart Rate'),
        alt.Tooltip('st_depression:Q', title='ST Depression'),
        alt.Tooltip('disease:N', title='Disease Status')
    ]
).add_params(
    brush
)

trend = alt.Chart(df_sample).transform_regression(
    'max_hr', 'st_depression', groupby=['disease']
).mark_line(size=3).encode(
    x='max_hr:Q',
    y='st_depression:Q',
    color=alt.Color(
        'disease:N',
        title='Disease Status',
        scale=alt.Scale(domain=['No Disease', 'Disease'])
    )
)

scatter_with_trend = (points + trend).properties(
    title='Maximum Heart Rate vs ST Depression by Disease Status',
    width=500,
    height=350
)

scatter_with_trend
